# Task 2: XGBoost Benchmark & MLP Comparison — Same 8:1:1 Split

**Goal:** Exactly replicate Task 16's XGBoost 64D pipeline, add XGBoost 70D, and compare against MLP 64D/70D on the same held-out test set with paired gene-cluster bootstrap.

Key: Task 16 does NOT apply StandardScaler to DDG/TMD features — only SimpleImputer. ESM features get both imputation and scaling before PCA.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

ROOT = Path("/mnt/volume6/czj/labLGN/LabLZ")
BASE = ROOT / "xgboost_trial"
MLP_TRIAL = ROOT / "mlp_trial"
MLP_TRIAL.mkdir(parents=True, exist_ok=True)

FEATURES_CSV = BASE / "features_v3.csv"
TMD_CSV = BASE / "tmd_features.csv"
CONTEXT_CSV = BASE / "deeploc_wt_context_features.csv"
SPLIT_CSV = BASE / "task16_holdout_split.csv"
MLP_PREDS_CSV = MLP_TRIAL / "task1_mlp_predictions.csv"

XGB_PREDS_CSV = MLP_TRIAL / "task2_xgboost_benchmark_predictions.csv"
BOOTSTRAP_CSV = MLP_TRIAL / "task2_test_bootstrap.csv"
BOOTSTRAP_SUMMARY_CSV = MLP_TRIAL / "task2_test_bootstrap_summary.csv"

RANDOM_STATE = 42
N_COMPONENTS = 50
STRUCT_COLS = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand", "ss_coil", "delta_hydrophobicity"]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_COLS = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]
WT_SIGNAL_COLS = [f"deeploc_wt_signal_{c}" for c in [
    "signal_peptide", "mitochondrial_transit_peptide",
    "nuclear_localisation_signal", "nuclear_export_signal",
    "peroxisomal_targeting_signal", "gpi_anchor"]]

## 2.1 Load data and reuse Task 16 split

In [ ]:
df = pd.read_csv(FEATURES_CSV)
assert len(df) == 2179 and df["KEY"].is_unique
assert df["Mislocalized"].isin([0, 1]).all() and int(df["Mislocalized"].sum()) == 236

# Merge additional features — exact same order as Task 16
for name in DDG_COLS:
    table = pd.read_csv(BASE / f"{name}.csv")
    assert table["KEY"].is_unique and name in table.columns
    df = df.merge(table[["KEY", name]], on="KEY", how="left", validate="one_to_one")

tmd = pd.read_csv(TMD_CSV)
assert tmd["KEY"].is_unique and all(c in tmd.columns for c in TMD_COLS)
df = df.merge(tmd[["KEY"] + TMD_COLS], on="KEY", how="left", validate="one_to_one")

context = pd.read_csv(CONTEXT_CSV)
assert context["KEY"].is_unique and all(c in context.columns for c in WT_SIGNAL_COLS)
df = df.merge(context[["KEY"] + WT_SIGNAL_COLS], on="KEY", how="left", validate="one_to_one")

split = pd.read_csv(SPLIT_CSV)
assert split["KEY"].is_unique and set(split["split"]) == {"train", "validation", "test"}
df = df.merge(split[["KEY", "split"]], on="KEY", how="left", validate="one_to_one")

esm_cols = [c for c in df.columns if c.startswith("esm_")]
assert len(esm_cols) == 1280

y = df["Mislocalized"].astype(int).to_numpy()
split_indices = {name: np.flatnonzero(df["split"].to_numpy() == name)
                 for name in ["train", "validation", "test"]}
train_idx, val_idx, test_idx = split_indices["train"], split_indices["validation"], split_indices["test"]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

gene_sets = {name: set(df.loc[idx, "Gene"].astype(str)) for name, idx in split_indices.items()}
assert gene_sets["train"].isdisjoint(gene_sets["validation"])
assert gene_sets["train"].isdisjoint(gene_sets["test"])
assert gene_sets["validation"].isdisjoint(gene_sets["test"])

for name, idx in split_indices.items():
    print(f"{name:10s}: n={len(idx):4d}, genes={len(gene_sets[name]):3d}, positives={int(y[idx].sum()):3d}, prevalence={y[idx].mean():.4f}")

## 2.2 Preprocessing — exact Task 16 replication

**Critical difference from MLP pipeline:** Task 16 does NOT apply StandardScaler to DDG/TMD — only SimpleImputer. ESM gets both imputation and scaling before PCA. Structure features get imputation + scaling.

In [ ]:
X_esm = df[esm_cols].to_numpy(np.float32)
X_struct = df[STRUCT_COLS].to_numpy(np.float32)
X_ddg = df[DDG_COLS].to_numpy(np.float32)
X_tmd = df[TMD_COLS].to_numpy(np.float32)
X_signal = df[WT_SIGNAL_COLS].to_numpy(np.float32)

def prepare_features(train_idx, val_idx, test_idx, include_signals=False):
    """Exact Task 16 preprocessing: ESM gets impute+scale+PCA; struct gets impute+scale;
    DDG/TMD get impute ONLY (no StandardScaler). Signals get impute ONLY (probabilities in [0,1])."""
    
    # ESM: impute -> scale -> PCA
    esm_imputer = SimpleImputer(strategy="median")
    esm_scaler = StandardScaler()
    esm_train = esm_scaler.fit_transform(esm_imputer.fit_transform(X_esm[train_idx]))
    esm_val   = esm_scaler.transform(esm_imputer.transform(X_esm[val_idx]))
    esm_test  = esm_scaler.transform(esm_imputer.transform(X_esm[test_idx]))
    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pc_train, pc_val, pc_test = pca.fit_transform(esm_train), pca.transform(esm_val), pca.transform(esm_test)

    # Structure: impute -> scale
    struct_imputer = SimpleImputer(strategy="median")
    struct_scaler = StandardScaler()
    struct_train = struct_scaler.fit_transform(struct_imputer.fit_transform(X_struct[train_idx]))
    struct_val   = struct_scaler.transform(struct_imputer.transform(X_struct[val_idx]))
    struct_test  = struct_scaler.transform(struct_imputer.transform(X_struct[test_idx]))

    # DDG + TMD: impute ONLY (Task 16 does NOT scale these)
    extra_imputer = SimpleImputer(strategy="median")
    extra_train_raw = np.hstack([X_ddg[train_idx], X_tmd[train_idx]])
    extra_val_raw   = np.hstack([X_ddg[val_idx],   X_tmd[val_idx]])
    extra_test_raw  = np.hstack([X_ddg[test_idx],  X_tmd[test_idx]])
    extra_train = extra_imputer.fit_transform(extra_train_raw)
    extra_val   = extra_imputer.transform(extra_val_raw)
    extra_test  = extra_imputer.transform(extra_test_raw)

    train = np.hstack([pc_train, struct_train, extra_train]).astype(np.float32)
    val   = np.hstack([pc_val,   struct_val,   extra_val]).astype(np.float32)
    test  = np.hstack([pc_test,  struct_test,  extra_test]).astype(np.float32)
    
    if include_signals:
        # Signal features: impute ONLY (probabilities in [0,1], no scaling needed)
        signal_imputer = SimpleImputer(strategy="median")
        signal_train = signal_imputer.fit_transform(X_signal[train_idx])
        signal_val   = signal_imputer.transform(X_signal[val_idx])
        signal_test  = signal_imputer.transform(X_signal[test_idx])
        train = np.hstack([train, signal_train]).astype(np.float32)
        val   = np.hstack([val,   signal_val]).astype(np.float32)
        test  = np.hstack([test,  signal_test]).astype(np.float32)

    expected_dim = 64 + (6 if include_signals else 0)
    assert train.shape[1] == val.shape[1] == test.shape[1] == expected_dim
    assert np.isfinite(train).all() and np.isfinite(val).all() and np.isfinite(test).all()
    return train, val, test

X64_train, X64_val, X64_test = prepare_features(train_idx, val_idx, test_idx, include_signals=False)
X70_train, X70_val, X70_test = prepare_features(train_idx, val_idx, test_idx, include_signals=True)
print(f"64D: train={X64_train.shape}, val={X64_val.shape}, test={X64_test.shape}")
print(f"70D: train={X70_train.shape}, val={X70_val.shape}, test={X70_test.shape}")

## 2.3 Train XGBoost 64D and 70D — exact Task 16 hyperparameters

In [ ]:
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

results = {}
for dim_name, X_tr, X_va, X_te in [
    ("xgb_64", X64_train, X64_val, X64_test),
    ("xgb_70", X70_train, X70_val, X70_test),
]:
    xgb = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.5, objective="binary:logistic", eval_metric="aucpr",
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )
    xgb.fit(X_tr, y_train, sample_weight=sample_weight, verbose=False)
    val_prob = xgb.predict_proba(X_va)[:, 1]
    test_prob = xgb.predict_proba(X_te)[:, 1]
    results[dim_name] = {"val_prob": val_prob, "test_prob": test_prob}
    print(f"{dim_name}: val AUROC={roc_auc_score(y_val, val_prob):.4f}, val AUPRC={average_precision_score(y_val, val_prob):.4f}, "
          f"test AUROC={roc_auc_score(y_test, test_prob):.4f}, test AUPRC={average_precision_score(y_test, test_prob):.4f}")

# Verify: XGBoost 64D should match Task 16's AUROC=0.6266, AUPRC=0.2075
xgb64_auroc = roc_auc_score(y_test, results["xgb_64"]["test_prob"])
xgb64_auprc = average_precision_score(y_test, results["xgb_64"]["test_prob"])
assert abs(xgb64_auroc - 0.6266) < 0.001, f"XGB64 AUROC mismatch: {xgb64_auroc:.4f} vs expected 0.6266"
assert abs(xgb64_auprc - 0.2075) < 0.001, f"XGB64 AUPRC mismatch: {xgb64_auprc:.4f} vs expected 0.2075"
print("\nXGBoost 64D reproduces Task 16 results.")

## 2.4 Load MLP predictions and compare

In [ ]:
mlp_preds = pd.read_csv(MLP_PREDS_CSV)
mlp_test = mlp_preds[mlp_preds["split"] == "test"]
mlp_64_test = mlp_test["mlp_64d_probability"].to_numpy()
mlp_70_test = mlp_test["mlp_70d_probability"].to_numpy()

all_models = {
    "XGB64": results["xgb_64"]["test_prob"],
    "XGB70": results["xgb_70"]["test_prob"],
    "MLP64": mlp_64_test,
    "MLP70": mlp_70_test,
}

print("=== Test set metrics (210 variants, 23 positives) ===")
for name, score in all_models.items():
    print(f"  {name:10s}: AUROC={roc_auc_score(y_test, score):.4f}, AUPRC={average_precision_score(y_test, score):.4f}")

# Save XGBoost predictions
xgb_out = df.iloc[np.concatenate([val_idx, test_idx])][["KEY", "Gene", "Mutation_used", "Mislocalized", "split"]].copy()
xgb_out["xgb_64d_probability"] = np.nan
xgb_out["xgb_70d_probability"] = np.nan
xgb_out.loc[xgb_out["split"] == "validation", "xgb_64d_probability"] = results["xgb_64"]["val_prob"]
xgb_out.loc[xgb_out["split"] == "test",       "xgb_64d_probability"] = results["xgb_64"]["test_prob"]
xgb_out.loc[xgb_out["split"] == "validation", "xgb_70d_probability"] = results["xgb_70"]["val_prob"]
xgb_out.loc[xgb_out["split"] == "test",       "xgb_70d_probability"] = results["xgb_70"]["test_prob"]
xgb_out.to_csv(XGB_PREDS_CSV, index=False)
print(f"\nSaved: {XGB_PREDS_CSV}")

## 2.5 Paired gene-cluster bootstrap on test set

Resample genes with replacement (2,000 replicates). Compare all four models pairwise on the same 210-variant test set.

In [ ]:
test_df = df.iloc[test_idx].copy()
test_genes = test_df["Gene"].astype(str).to_numpy()
unique_test_genes = np.unique(test_genes)
gene_to_test_idx = {g: np.flatnonzero(test_genes == g) for g in unique_test_genes}

comparisons = [
    ("MLP64", "XGB64", "MLP64 vs XGB64"),
    ("MLP70", "XGB70", "MLP70 vs XGB70"),
    ("MLP70", "MLP64", "MLP70 vs MLP64"),
    ("XGB70", "XGB64", "XGB70 vs XGB64"),
    ("MLP64", "MLP70", "MLP64 vs MLP70"),
    ("XGB64", "XGB70", "XGB64 vs XGB70"),
]

rng = np.random.default_rng(42)
bootstrap_rows = []
for rep in range(2000):
    sampled = rng.choice(unique_test_genes, size=len(unique_test_genes), replace=True)
    idx = np.concatenate([gene_to_test_idx[g] for g in sampled])
    if np.unique(y_test[idx]).size < 2:
        continue
    for m1, m2, label in comparisons:
        s1, s2 = all_models[m1][idx], all_models[m2][idx]
        bootstrap_rows.append({
            "replicate": rep, "comparison": label,
            "delta_auroc": roc_auc_score(y_test[idx], s1) - roc_auc_score(y_test[idx], s2),
            "delta_auprc": average_precision_score(y_test[idx], s1) - average_precision_score(y_test[idx], s2),
        })

bootstrap = pd.DataFrame(bootstrap_rows)
summary = bootstrap.groupby("comparison").agg(
    delta_auroc_mean=("delta_auroc", "mean"),
    delta_auroc_low=("delta_auroc", lambda x: np.quantile(x, 0.025)),
    delta_auroc_high=("delta_auroc", lambda x: np.quantile(x, 0.975)),
    delta_auprc_mean=("delta_auprc", "mean"),
    delta_auprc_low=("delta_auprc", lambda x: np.quantile(x, 0.025)),
    delta_auprc_high=("delta_auprc", lambda x: np.quantile(x, 0.975)),
).reset_index()

print("=== Paired gene-cluster bootstrap (test set, 210 variants, 23 pos, 2000 reps) ===")
print(summary.to_string(index=False))

bootstrap.to_csv(BOOTSTRAP_CSV, index=False)
summary.to_csv(BOOTSTRAP_SUMMARY_CSV, index=False)
print(f"\nSaved: {BOOTSTRAP_CSV}")
print(f"Saved: {BOOTSTRAP_SUMMARY_CSV}")

## Interpretation

This is a single held-out test with 23 positives — CIs will be wide. Directionally consistent point estimates that favour one model family (MLP or XGBoost) would justify five-fold OOF evaluation. Seeing the test results and then tuning any model invalidates the test as independent.